# Silver Layer - Data Transformation

This notebook transforms the raw Bronze Delta data into
cleaned and business-ready Silver datasets.

## What we do

- Read Bronze Delta data.
- Validate and clean records.
- Handle missing values.
- Remove duplicate records.
- Apply basic transformations.
- Join related datasets.
- Write the transformed data back as Delta tables.

## Architecture

```text
Bronze Delta
     ↓
Cleaning
     ↓
Validation
     ↓
Transformation
     ↓
Joins
     ↓
Silver Delta
```

The Silver layer contains cleaned and structured data that can be used for analytics and downstream processing.

In [0]:
DATA_PATH = "/Volumes/workspace/default/olist_data"

BRONZE_PATH = f"{DATA_PATH}/bronze"

orders_df = spark.read.format("delta").load(
    f"{BRONZE_PATH}/orders"
)

customers_df = spark.read.format("delta").load(
    f"{BRONZE_PATH}/customers"
)

products_df = spark.read.format("delta").load(
    f"{BRONZE_PATH}/products"
)

order_items_df = spark.read.format("delta").load(
    f"{BRONZE_PATH}/order_items"
)

payments_df = spark.read.format("delta").load(
    f"{BRONZE_PATH}/payments"
)

In [0]:
print("Orders:", orders_df.count())
print("Customers:", customers_df.count())
print("Products:", products_df.count())
print("Order Items:", order_items_df.count())
print("Payments:", payments_df.count())

In [0]:
orders_df.select("order_id").distinct().count()

In [0]:
silver_orders_df = (
    orders_df
    .dropDuplicates(["order_id"])
    .filter("order_id IS NOT NULL")
    .filter("customer_id IS NOT NULL")
)

In [0]:
from pyspark.sql.functions import to_date

silver_orders_df = silver_orders_df.withColumn(
    "order_date",
    to_date("order_purchase_timestamp")
)

In [0]:
display(
    silver_orders_df.select(
        "order_id",
        "customer_id",
        "order_status",
        "order_purchase_timestamp",
        "order_date"
    )
)

In [0]:
silver_orders_customers_df = (
    silver_orders_df
    .join(
        customers_df,
        on="customer_id",
        how="left"
    )
)

In [0]:
display(silver_orders_customers_df)

In [0]:
silver_order_details_df = (
    silver_orders_customers_df
    .join(
        order_items_df,
        on="order_id",
        how="left"
    )
)

In [0]:
display(silver_order_details_df)

In [0]:
silver_order_details_df = (
    silver_order_details_df
    .join(
        products_df,
        on="product_id",
        how="left"
    )
)

In [0]:
display(
    silver_order_details_df.select(
        "order_id",
        "customer_id",
        "product_id",
        "order_status",
        "order_purchase_timestamp"
    )
)

In [0]:
print("Silver orders:", silver_orders_df.count())

print(
    "Orders + Customers:",
    silver_orders_customers_df.count()
)

print(
    "Orders + Customers + Items:",
    silver_order_details_df.count()
)

print(
    "Orders + Customers + Items + Products:",
    silver_order_details_df.count()
)

In [0]:
from pyspark.sql.functions import sum, count

payments_agg_df = (
    payments_df
    .groupBy("order_id")
    .agg(
        sum("payment_value").alias("total_payment_value"),
        count("*").alias("payment_count")
    )
)

In [0]:
display(payments_agg_df)

In [0]:
silver_order_details_df = (
    silver_order_details_df
    .join(
        payments_agg_df,
        on="order_id",
        how="left"
    )
)

In [0]:
print(
    "Final Silver rows:",
    silver_order_details_df.count()
)

In [0]:
silver_order_details_df = silver_order_details_df.select(
    "order_id",
    "customer_id",
    "customer_unique_id",
    "order_status",
    "order_purchase_timestamp",
    "order_date",
    "product_id",
    "product_category_name",
    "seller_id",
    "price",
    "freight_value",
    "total_payment_value",
    "payment_count"
)

In [0]:
display(silver_order_details_df)

In [0]:
SILVER_PATH = f"{DATA_PATH}/silver/order_details"

(
    silver_order_details_df.write
    .format("delta")
    .mode("overwrite")
    .save(SILVER_PATH)
)

In [0]:
silver_check_df = (
    spark.read
    .format("delta")
    .load(SILVER_PATH)
)

print("Silver rows:", silver_check_df.count())
display(silver_check_df)